
# NB_02 — Customers Incremental Bronze-to-Silver Load

## Purpose

This notebook incrementally processes **Customer** data from the Bronze
Lakehouse into the Silver Lakehouse.

Unlike a full load, this notebook processes only customer records that are
newer than the last successfully processed watermark.


| Component | Object |
|---|---|
| Source | `LH_Bronze.dbo.bronze_customers` |
| Target | `LH_Silver.dbo.silver_customers` |
| Control | `LH_Silver.dbo.etl_control` |
| Audit | `LH_Silver.dbo.etl_batch_audit` |
| Business Key | `customer_id` |
| Watermark | `created_date` |

---

## Architecture Position

```text
                    LH_Silver.dbo.etl_control
                              |
                       Read CUSTOMER
                         watermark
                              |
                              v
LH_Bronze                                     LH_Silver
bronze_customers                              silver_customers
      |                                              ^
      |                                              |
      +----> Incremental Filter                      |
                    |                                |
                    v                                |
              Data Validation                       |
                    |                                |
                    v                                |
               Transformation                       |
                    |                                |
                    v                                |
              Delta MERGE --------------------------+
                    |
                    +----> etl_batch_audit
                    |
                    +----> Update CUSTOMER watermark


                    



## Notebook at a Glance

This notebook is the reference implementation for metadata-driven incremental
Bronze-to-Silver processing in the Insurance Medallion architecture.

### Processing Flow

`Bronze Customers`
→ `Read ETL Control`
→ `Read Watermark`
→ `Incremental Filter`
→ `Data Quality Validation`
→ `Deduplication`
→ `Transformation`
→ `Delta MERGE`
→ `Silver Customers`
→ `Audit Execution`
→ `Advance Watermark`

### Configuration

| Component | Value |
|---|---|
| Source | `LH_Bronze.dbo.bronze_customers` |
| Target | `LH_Silver.dbo.silver_customers` |
| Business Key | `customer_id` |
| Watermark | `created_date` |
| Control Table | `LH_Silver.dbo.etl_control` |
| Audit Table | `LH_Silver.dbo.etl_batch_audit` |
| Load Type | `INCREMENTAL` |
| Storage | Delta |
| Write Pattern | Delta MERGE |

### Execution Outcomes

**SUCCESS**  
New data → Validate → Transform → MERGE → Audit → Advance watermark

**NO_DATA**  
No new data → Audit `NO_DATA` → Keep watermark unchanged

**FAILED**  
Processing error → Audit `FAILED` → Capture error → Keep watermark unchanged

> **Core design principle:** The watermark advances only after successful
> Silver processing.


## Step 2 — Initialize Customer Incremental Processing

This step initializes the Customer processing notebook.

The notebook defines the Customer entity being processed and references the
ETL control and audit tables created by `NB_01_ETL_Control_Framework`.

The actual source, target, watermark column, and current watermark will be
retrieved from `etl_control` rather than duplicated as processing configuration
inside this notebook.


In [1]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# ---------------------------------------------------------
# CUSTOMER INCREMENTAL PROCESSING CONFIGURATION
# ---------------------------------------------------------

SOURCE_NAME = "CUSTOMERS"

CONTROL_TABLE = "LH_Silver.dbo.etl_control"
AUDIT_TABLE   = "LH_Silver.dbo.etl_batch_audit"

PIPELINE_NAME = "PL_Insurance_Medallion_ETL"

# Unique execution identifier
BATCH_ID = str(uuid.uuid4())

print("Customer incremental processing initialized.")
print(f"Source name : {SOURCE_NAME}")
print(f"Batch ID    : {BATCH_ID}")
print(f"Pipeline    : {PIPELINE_NAME}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 3, Finished, Available, Finished, False)

Customer incremental processing initialized.
Source name : CUSTOMERS
Batch ID    : e03edb71-015d-452f-aadd-c088ce829d2b
Pipeline    : PL_Insurance_Medallion_ETL



## Step 3 — Read Customer Configuration from ETL Control

The notebook does not hard-code the Bronze source table, Silver target table,
or watermark configuration.

Instead, it retrieves the active `CUSTOMERS` configuration from:

`LH_Silver.dbo.etl_control`

This allows processing behavior to be controlled through metadata.

The configuration provides:

- Bronze source table
- Silver target table
- Watermark column
- Last successfully processed watermark
- Load type
- Active/inactive status

This is the foundation of the metadata-driven incremental processing pattern.

In [2]:

# ---------------------------------------------------------
# READ CUSTOMER CONFIGURATION FROM ETL CONTROL
# ---------------------------------------------------------

customer_config_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)

config_count = customer_config_df.count()

if config_count != 1:
    raise ValueError(
        f"Expected exactly one active configuration for {SOURCE_NAME}, "
        f"but found {config_count}."
    )

display(customer_config_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ebd64bc7-f308-4f90-bc79-64fdb9677314)


## Step 4 — Extract Runtime Processing Configuration

The Customer configuration has been retrieved from `etl_control`.

This step converts the metadata row into runtime variables used by the
incremental processing logic.

The notebook therefore does not need to hard-code:

- Source table
- Target table
- Watermark column
- Last processed watermark
- Load type

These values are supplied by the ETL control framework at runtime.

In [3]:

# ---------------------------------------------------------
# EXTRACT RUNTIME CONFIGURATION
# ---------------------------------------------------------

config = customer_config_df.first()

SOURCE_TABLE     = config["source_table"]
TARGET_TABLE     = config["target_table"]
WATERMARK_COLUMN = config["watermark_column"]
LAST_WATERMARK   = config["last_watermark"]
LOAD_TYPE        = config["load_type"]

print("Runtime configuration loaded.")
print("--------------------------------------------")
print(f"Source name      : {SOURCE_NAME}")
print(f"Source table     : {SOURCE_TABLE}")
print(f"Target table     : {TARGET_TABLE}")
print(f"Watermark column : {WATERMARK_COLUMN}")
print(f"Last watermark   : {LAST_WATERMARK}")
print(f"Load type        : {LOAD_TYPE}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 5, Finished, Available, Finished, False)

Runtime configuration loaded.
--------------------------------------------
Source name      : CUSTOMERS
Source table     : LH_Bronze.dbo.bronze_customers
Target table     : LH_Silver.dbo.silver_customers
Watermark column : created_date
Last watermark   : 1900-01-01 00:00:00
Load type        : INCREMENTAL



## Step 5 — Read Incremental Customer Records

This step reads the Customer Bronze table using the source table obtained
from `etl_control`.

The current watermark is then applied to identify records that have not yet
been successfully processed.

### Incremental Rule

For Customers:

`created_date > last_watermark`

Only records satisfying this condition are included in the current batch.

The maximum watermark found in this batch will be captured for use after
successful Silver processing.

**Important:** The control-table watermark is not updated at this stage.
It is advanced only after the Silver write completes successfully.

In [4]:

# ---------------------------------------------------------
# READ BRONZE AND APPLY INCREMENTAL WATERMARK
# ---------------------------------------------------------

bronze_df = spark.table(SOURCE_TABLE)

source_total_count = bronze_df.count()

incremental_df = (
    bronze_df
    .filter(
        F.to_timestamp(F.col(WATERMARK_COLUMN))
        > F.lit(LAST_WATERMARK)
    )
)

incremental_count = incremental_df.count()

print("Bronze customer data read.")
print("--------------------------------------------")
print(f"Source table        : {SOURCE_TABLE}")
print(f"Total Bronze rows   : {source_total_count}")
print(f"Last watermark      : {LAST_WATERMARK}")
print(f"Incremental rows    : {incremental_count}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 6, Finished, Available, Finished, False)

Bronze customer data read.
--------------------------------------------
Source table        : LH_Bronze.dbo.bronze_customers
Total Bronze rows   : 501
Last watermark      : 1900-01-01 00:00:00
Incremental rows    : 501


In [5]:
display(
    incremental_df
    .orderBy(F.col(WATERMARK_COLUMN).desc())
    .limit(10)
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 912de93d-9886-4653-a26c-ee6fdadc33fb)


## Step 6 — Validate Incremental Customer Data

Before Customer records are written to Silver, the incremental dataset is
validated for conditions that could make the load unreliable.

### Validation Rules

For this Customer load:

- `customer_id` must not be null.
- `customer_id` must not be duplicated within the incremental batch.
- `created_date` must contain a valid date.
- Records failing these checks are rejected from Silver processing.

The validation step separates the incoming batch into:

`valid_customer_df` — records eligible for Silver processing.

`rejected_customer_df` — records that failed data-quality validation.

Reject counts are captured later in `etl_batch_audit`.

In [6]:

# ---------------------------------------------------------
# VALIDATE INCREMENTAL CUSTOMER DATA
# ---------------------------------------------------------

# Add parsed watermark/date column
validated_df = (
    incremental_df
    .withColumn(
        "_parsed_created_date",
        F.to_timestamp(F.col(WATERMARK_COLUMN))
    )
)

# Detect duplicate business keys within this batch
duplicate_keys_df = (
    validated_df
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .select("customer_id")
)

# Reject records with:
# 1. Missing customer_id
# 2. Invalid/null created_date
# 3. Duplicate customer_id in current batch

rejected_customer_df = (
    validated_df.alias("src")
    .join(
        duplicate_keys_df.alias("dup"),
        F.col("src.customer_id") == F.col("dup.customer_id"),
        "left"
    )
    .filter(
        F.col("src.customer_id").isNull()
        | F.col("src._parsed_created_date").isNull()
        | F.col("dup.customer_id").isNotNull()
    )
    .select("src.*")
)

valid_customer_df = (
    validated_df.alias("src")
    .join(
        duplicate_keys_df.alias("dup"),
        F.col("src.customer_id") == F.col("dup.customer_id"),
        "left"
    )
    .filter(
        F.col("src.customer_id").isNotNull()
        & F.col("src._parsed_created_date").isNotNull()
        & F.col("dup.customer_id").isNull()
    )
    .select("src.*")
)

valid_count  = valid_customer_df.count()
reject_count = rejected_customer_df.count()

print("Customer data-quality validation completed.")
print("--------------------------------------------")
print(f"Incremental records : {incremental_count}")
print(f"Valid records       : {valid_count}")
print(f"Rejected records    : {reject_count}")
print(f"Reconciliation      : {valid_count + reject_count}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 8, Finished, Available, Finished, False)

Customer data-quality validation completed.
--------------------------------------------
Incremental records : 501
Valid records       : 499
Rejected records    : 2
Reconciliation      : 501



### Step 6A — Inspect Rejected Customer Records

Two Customer records failed the data-quality rules.

Before continuing to Silver processing, the rejected records are inspected
to determine which validation rule caused the rejection.

In [7]:

# ---------------------------------------------------------
# INSPECT REJECTED CUSTOMER RECORDS
# ---------------------------------------------------------

display(
    rejected_customer_df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "created_date",
        "_parsed_created_date"
    )
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 939ac2c7-cb80-4a5c-a9da-b04ab2405cee)

In [8]:

# ---------------------------------------------------------
# SHOW VALIDATION FAILURE REASONS
# ---------------------------------------------------------

reject_analysis_df = (
    validated_df.alias("src")
    .join(
        duplicate_keys_df.alias("dup"),
        F.col("src.customer_id") == F.col("dup.customer_id"),
        "left"
    )
    .filter(
        F.col("src.customer_id").isNull()
        | F.col("src._parsed_created_date").isNull()
        | F.col("dup.customer_id").isNotNull()
    )
    .select(
        "src.customer_id",
        "src.first_name",
        "src.last_name",
        "src.created_date",
        F.when(
            F.col("src.customer_id").isNull(),
            "MISSING_CUSTOMER_ID"
        ).when(
            F.col("src._parsed_created_date").isNull(),
            "INVALID_CREATED_DATE"
        ).when(
            F.col("dup.customer_id").isNotNull(),
            "DUPLICATE_CUSTOMER_ID"
        ).alias("reject_reason")
    )
)

display(reject_analysis_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8447e4aa-b3af-4466-ad48-c731cb21b426)


### Step 6B — Resolve Duplicate Customer Records

The incremental batch contains duplicate records for the same Customer
business key.

Rather than rejecting every occurrence of the duplicated customer, the
processing layer applies deterministic deduplication.

For each `customer_id`, one record is retained for Silver processing and
additional copies are treated as duplicate rejects.

This ensures that:

- Silver maintains one record per Customer business key.
- Valid customer information is not lost because of duplicate ingestion.
- Duplicate records remain measurable through audit metrics.
- Delta MERGE receives only one source row per business key.

In [9]:

# ---------------------------------------------------------
# DEDUPLICATE INCREMENTAL CUSTOMER RECORDS
# ---------------------------------------------------------

from pyspark.sql.window import Window

customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("_parsed_created_date").desc(),
        F.col("customer_id")
    )
)

ranked_customer_df = (
    validated_df
    .withColumn(
        "_duplicate_rank",
        F.row_number().over(customer_window)
    )
)

# Valid records:
# - business key exists
# - watermark/date is valid
# - first occurrence of the customer
valid_customer_df = (
    ranked_customer_df
    .filter(
        F.col("customer_id").isNotNull()
        & F.col("_parsed_created_date").isNotNull()
        & (F.col("_duplicate_rank") == 1)
    )
)

# Reject everything else
rejected_customer_df = (
    ranked_customer_df
    .filter(
        F.col("customer_id").isNull()
        | F.col("_parsed_created_date").isNull()
        | (F.col("_duplicate_rank") > 1)
    )
    .withColumn(
        "reject_reason",
        F.when(
            F.col("customer_id").isNull(),
            F.lit("MISSING_CUSTOMER_ID")
        )
        .when(
            F.col("_parsed_created_date").isNull(),
            F.lit("INVALID_CREATED_DATE")
        )
        .otherwise(
            F.lit("DUPLICATE_CUSTOMER_ID")
        )
    )
)

valid_count  = valid_customer_df.count()
reject_count = rejected_customer_df.count()

print("Customer deduplication completed.")
print("--------------------------------------------")
print(f"Incremental records : {incremental_count}")
print(f"Valid unique records: {valid_count}")
print(f"Rejected records    : {reject_count}")
print(f"Reconciliation      : {valid_count + reject_count}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 11, Finished, Available, Finished, False)

Customer deduplication completed.
--------------------------------------------
Incremental records : 501
Valid unique records: 500
Rejected records    : 1
Reconciliation      : 501


In [10]:
display(
    rejected_customer_df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "created_date",
        "_duplicate_rank",
        "reject_reason"
    )
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e85c1f46-355b-49a3-b564-1c84bd59185a)

## Step 7 — Prepare Customer Records for Silver

The validated and deduplicated Customer records are now prepared for the
Silver layer.

Bronze preserves source-oriented data, while Silver applies standardized
data types and adds processing metadata.

### Silver Transformations

For Customer records:

- `date_of_birth` is converted to a date.
- `created_date` is converted to a date.
- Customer business attributes are preserved.
- Temporary validation columns are removed.
- `silver_processed_ts` is added for operational traceability.

The resulting DataFrame will be used as the source for the Delta MERGE.

In [11]:

# ---------------------------------------------------------
# PREPARE CUSTOMER RECORDS FOR SILVER
# ---------------------------------------------------------

silver_ready_df = (
    valid_customer_df
    .select(
        F.col("customer_id"),
        F.col("first_name"),
        F.col("last_name"),

        F.to_date(F.col("date_of_birth"))
            .alias("date_of_birth"),

        F.col("email"),
        F.col("phone"),
        F.col("address"),
        F.col("city"),
        F.col("state"),
        F.col("zip_code"),

        F.to_date(F.col("created_date"))
            .alias("created_date"),

        F.col("customer_status"),

        F.current_timestamp()
            .alias("silver_processed_ts")
    )
)

silver_ready_count = silver_ready_df.count()

print("Customer Silver transformation completed.")
print("--------------------------------------------")
print(f"Valid input records : {valid_count}")
print(f"Silver-ready records: {silver_ready_count}")

assert silver_ready_count == valid_count, \
    "Silver transformation changed the expected record count."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 13, Finished, Available, Finished, False)

Customer Silver transformation completed.
--------------------------------------------
Valid input records : 500
Silver-ready records: 500


In [12]:

silver_ready_df.printSchema()

display(
    silver_ready_df
    .orderBy(F.col("created_date").desc())
    .limit(10)
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 14, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- silver_processed_ts: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, 4382ebbe-ea7b-4188-b663-2dde8ffa73ac)


## Step 8 — Delta MERGE into Silver

The validated Customer dataset is now ready to be synchronized with the
Silver Customer table using Delta Lake `MERGE`.

Before executing the MERGE, the notebook first compares the incoming
Customer business keys with the existing Silver table.

This allows us to determine how many incoming records represent:

- **INSERTS** — Customer does not currently exist in Silver.
- **MATCHES** — Customer already exists in Silver and is eligible for update.

The Customer business key used for matching is `customer_id`.

No Silver data is modified during this inspection step.

In [13]:

# ---------------------------------------------------------
# STEP 8A - INSPECT EXISTING SILVER CUSTOMER TABLE
# ---------------------------------------------------------

existing_silver_df = spark.table(TARGET_TABLE)

existing_silver_count = existing_silver_df.count()

print("Existing Silver Customer table inspected.")
print("--------------------------------------------")
print(f"Target table         : {TARGET_TABLE}")
print(f"Existing Silver rows : {existing_silver_count}")

existing_silver_df.printSchema()

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 15, Finished, Available, Finished, False)

Existing Silver Customer table inspected.
--------------------------------------------
Target table         : LH_Silver.dbo.silver_customers
Existing Silver rows : 500
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = true)



### Step 8B — Analyze Delta MERGE Impact

Before modifying the Silver table, the incoming Customer dataset is compared
with the existing Silver Customer table using the business key `customer_id`.

This pre-MERGE analysis determines:

- **Insert candidates** — incoming Customer IDs that do not exist in Silver.
- **Update candidates** — incoming Customer IDs that already exist in Silver.

This validation provides visibility into the expected MERGE behavior before
any Silver data is modified.

The reconciliation rule is:

`Silver-ready records = Insert candidates + Update candidates`

In [14]:

# ---------------------------------------------------------
# STEP 8B - ANALYZE EXPECTED DELTA MERGE IMPACT
# ---------------------------------------------------------

# Existing Silver business keys
silver_keys_df = (
    existing_silver_df
    .select("customer_id")
    .dropDuplicates()
)

# Incoming records whose business key does NOT exist in Silver
insert_candidates_df = (
    silver_ready_df.alias("src")
    .join(
        silver_keys_df.alias("tgt"),
        F.col("src.customer_id") == F.col("tgt.customer_id"),
        "left_anti"
    )
)

# Incoming records whose business key already exists in Silver
update_candidates_df = (
    silver_ready_df.alias("src")
    .join(
        silver_keys_df.alias("tgt"),
        F.col("src.customer_id") == F.col("tgt.customer_id"),
        "left_semi"
    )
)

insert_count = insert_candidates_df.count()
update_count = update_candidates_df.count()

print("Delta MERGE impact analysis completed.")
print("--------------------------------------------")
print(f"Silver-ready records : {silver_ready_count}")
print(f"Insert candidates    : {insert_count}")
print(f"Update candidates    : {update_count}")
print(f"Reconciliation       : {insert_count + update_count}")

assert insert_count + update_count == silver_ready_count, \
    "MERGE impact reconciliation failed."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 16, Finished, Available, Finished, False)

Delta MERGE impact analysis completed.
--------------------------------------------
Silver-ready records : 500
Insert candidates    : 0
Update candidates    : 500
Reconciliation       : 500



                  500 incoming customers
                           │
                    customer_id match
                    /             \
                  NO               YES
                  │                 │
               INSERT       Compare attributes
                                  /     \
                              changed   same
                                 │       │
                              UPDATE   NO-OP


### Step 8C — Detect Changed and Unchanged Customer Records

A matching `customer_id` does not automatically mean that the Silver record
needs to be updated.

The incoming Customer record is compared with the existing Silver record
using the business attributes.

Matched records are classified as:

- **Changed** — one or more business attributes differ from Silver.
- **Unchanged** — the incoming record is identical to the current Silver record.

Only changed records should be updated.

This avoids unnecessary Delta Lake rewrites and reduces compute,
transaction-log activity, and downstream change processing.

The operational column `silver_processed_ts` is intentionally excluded
from the comparison because it represents processing metadata rather than
Customer business data.

In [15]:

# ---------------------------------------------------------
# STEP 8C - DETECT ACTUAL CHANGES
# ---------------------------------------------------------

compare_columns = [
    "first_name",
    "last_name",
    "date_of_birth",
    "email",
    "phone",
    "address",
    "city",
    "state",
    "zip_code",
    "created_date",
    "customer_status"
]

matched_df = (
    silver_ready_df.alias("src")
    .join(
        existing_silver_df.alias("tgt"),
        F.col("src.customer_id") == F.col("tgt.customer_id"),
        "inner"
    )
)

# Null-safe comparison:
# eqNullSafe treats NULL = NULL as equal.
change_condition = None

for column_name in compare_columns:

    column_changed = ~F.col(
        f"src.{column_name}"
    ).eqNullSafe(
        F.col(f"tgt.{column_name}")
    )

    change_condition = (
        column_changed
        if change_condition is None
        else change_condition | column_changed
    )

changed_customer_df = (
    matched_df
    .filter(change_condition)
    .select("src.*")
)

unchanged_customer_df = (
    matched_df
    .filter(~change_condition)
    .select("src.*")
)

changed_count = changed_customer_df.count()
unchanged_count = unchanged_customer_df.count()

print("Customer change detection completed.")
print("--------------------------------------------")
print(f"Matched customers   : {update_count}")
print(f"Changed customers   : {changed_count}")
print(f"Unchanged customers : {unchanged_count}")
print(f"Reconciliation      : {changed_count + unchanged_count}")

assert changed_count + unchanged_count == update_count, \
    "Change-detection reconciliation failed."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 17, Finished, Available, Finished, False)

Customer change detection completed.
--------------------------------------------
Matched customers   : 500
Changed customers   : 0
Unchanged customers : 500
Reconciliation      : 500



### Step 8D — Execute the Customer Delta MERGE

The incoming Customer data has now been classified into:

- New customers requiring INSERT
- Existing customers with changed attributes requiring UPDATE
- Existing customers with no changes requiring NO-OP

The Delta MERGE synchronizes the Silver Customer table using `customer_id`
as the business key.

Only changed matched records should be updated.

Unchanged records are intentionally left untouched.

In [16]:

# ---------------------------------------------------------
# STEP 8D - EXECUTE CUSTOMER DELTA MERGE
# ---------------------------------------------------------

target_delta = DeltaTable.forName(
    spark,
    TARGET_TABLE
)

merge_source_df = (
    insert_candidates_df
    .unionByName(
        changed_customer_df,
        allowMissingColumns=True
    )
)

merge_source_count = merge_source_df.count()

print("Preparing Customer Delta MERGE.")
print("--------------------------------------------")
print(f"Insert records : {insert_count}")
print(f"Changed records: {changed_count}")
print(f"MERGE source   : {merge_source_count}")

if merge_source_count > 0:

    (
        target_delta.alias("target")
        .merge(
            merge_source_df.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Customer Delta MERGE completed.")

else:

    print("No Customer inserts or updates detected.")
    print("Delta MERGE skipped.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 18, Finished, Available, Finished, False)

Preparing Customer Delta MERGE.
--------------------------------------------
Insert records : 0
Changed records: 0
MERGE source   : 0
No Customer inserts or updates detected.
Delta MERGE skipped.



## Step 9 — Write Customer Batch Audit Record

After Customer processing completes, an operational audit record is written
to `etl_batch_audit`.

The audit record captures the outcome of this Customer execution, including:

- Batch identifier
- Pipeline name
- Source record count
- Silver insert count
- Silver update count
- Rejected record count
- Processing status
- Start and completion timestamps
- Error information, when applicable

For this execution, the Silver MERGE was skipped because all existing Customer
records were unchanged. This is still considered a successful execution.

The audit table provides operational observability and allows pipeline runs
to be reconciled and investigated.

In [18]:

# ---------------------------------------------------------
# STEP 9A - WRITE CUSTOMER BATCH AUDIT RECORD
# ---------------------------------------------------------

from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)

audit_end_time = datetime.now()

audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), True),
    StructField("source_count", LongType(), False),
    StructField("insert_count", LongType(), False),
    StructField("update_count", LongType(), False),
    StructField("reject_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

audit_record = [
    (
        BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        datetime.now(),          # start_time
        audit_end_time,          # end_time
        incremental_count,       # source_count = 501
        insert_count,            # insert_count = 0
        changed_count,           # update_count = 0
        reject_count,            # reject_count = 1
        "SUCCESS",
        None                     # error_message
    )
]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)
display(audit_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bad676b3-57ff-438b-bd14-e286bb0e6e0b)


### Step 9B — Persist the Customer Audit Record

The audit record has been validated in memory.

This step appends the execution result to:

`LH_Silver.dbo.etl_batch_audit`

Each Customer execution therefore leaves an operational history record that can
be used for monitoring, reconciliation, and troubleshooting.

In [19]:
# ---------------------------------------------------------
# STEP 9B - PERSIST CUSTOMER AUDIT RECORD
# ---------------------------------------------------------

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print("Customer audit record written successfully.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 21, Finished, Available, Finished, False)

Customer audit record written successfully.


In [20]:
display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa28018f-05e2-4171-bf27-3591860b7513)

## Step 10 — Advance the Customer Watermark

The Customer load has completed successfully and its execution has been
recorded in `etl_batch_audit`.

The final step is to advance the Customer watermark in `etl_control`.

The watermark represents the latest successfully processed source position.

For Customers, the watermark column is:

`created_date`

The watermark is advanced only after:

1. Incremental records are identified.
2. Data-quality validation succeeds.
3. Duplicate records are handled.
4. Silver processing succeeds.
5. The batch audit record is successfully persisted.

If processing fails before these steps complete, the existing watermark
remains unchanged so that the records can be processed again safely.

This ordering provides restartability and prevents data loss.

In [21]:

# ---------------------------------------------------------
# STEP 10A - CALCULATE NEW CUSTOMER WATERMARK
# ---------------------------------------------------------

new_watermark = (
    valid_customer_df
    .agg(
        F.max(
            F.col("_parsed_created_date")
        ).alias("new_watermark")
    )
    .first()["new_watermark"]
)

print("Customer watermark calculated.")
print("--------------------------------------------")
print(f"Previous watermark : {LAST_WATERMARK}")
print(f"New watermark      : {new_watermark}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 23, Finished, Available, Finished, False)

Customer watermark calculated.
--------------------------------------------
Previous watermark : 1900-01-01 00:00:00
New watermark      : 2025-12-19 00:00:00



### Step 10B — Persist the New Customer Watermark

The maximum successfully processed Customer watermark has been calculated.

The Customer record in `etl_control` is now updated from the previous
watermark to the new successfully processed position.

This update occurs only after Silver processing and audit persistence
have completed successfully.

The next Customer incremental execution will therefore process only records
where:

`created_date > 2025-12-19 00:00:00`

This makes the incremental process restartable and prevents already processed
Customer records from being scanned again.

In [22]:

# ---------------------------------------------------------
# STEP 10B - UPDATE CUSTOMER WATERMARK
# ---------------------------------------------------------

control_delta = DeltaTable.forName(
    spark,
    CONTROL_TABLE
)

if new_watermark is not None:

    (
        control_delta.alias("target")
        .update(
            condition=(
                (F.col("target.source_name") == SOURCE_NAME)
                & (F.col("target.is_active") == True)
            ),
            set={
                "last_watermark": F.lit(new_watermark),
                "_updated_ts": F.current_timestamp()
            }
        )
    )

    print("Customer watermark updated successfully.")

else:
    print("No valid Customer records were processed.")
    print("Customer watermark was not changed.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 24, Finished, Available, Finished, False)

Customer watermark updated successfully.


In [23]:

# ---------------------------------------------------------
# STEP 10C - VERIFY CUSTOMER WATERMARK
# ---------------------------------------------------------

customer_control_check_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .select(
        "source_name",
        "watermark_column",
        "last_watermark",
        "load_type",
        "is_active",
        "_updated_ts"
    )
)

display(customer_control_check_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45e09f03-73e6-4936-acc1-af2b19dde829)

## Step 11 — Validate Incremental Restart Behavior

The Customer watermark has now been advanced to the latest successfully
processed `created_date`.

Before introducing new source data, the incremental filter is executed again
against the existing Bronze Customer table.

Because no Customer records have been added after the stored watermark,
the expected result is zero incremental records.

This validates an important property of the framework:

**Previously processed source records are not reprocessed during the next
incremental execution.**

Expected condition:

`created_date > last_watermark`

With the current watermark:

`created_date > 2025-12-19 00:00:00`

the expected incremental record count is `0`.

In [24]:

# ---------------------------------------------------------
# STEP 11A - READ WATERMARK FOR NEXT EXECUTION
# ---------------------------------------------------------

next_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

NEXT_WATERMARK = next_config["last_watermark"]

print("Next execution configuration loaded.")
print("--------------------------------------------")
print(f"Source name      : {SOURCE_NAME}")
print(f"Watermark column : {next_config['watermark_column']}")
print(f"Stored watermark : {NEXT_WATERMARK}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 26, Finished, Available, Finished, False)

Next execution configuration loaded.
--------------------------------------------
Source name      : CUSTOMERS
Watermark column : created_date
Stored watermark : 2025-12-19 00:00:00


In [25]:

# ---------------------------------------------------------
# STEP 11B - TEST NEXT INCREMENTAL READ
# ---------------------------------------------------------

next_bronze_df = spark.table(SOURCE_TABLE)

next_incremental_df = (
    next_bronze_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") > F.lit(NEXT_WATERMARK)
    )
)

next_incremental_count = next_incremental_df.count()

print("Next Customer incremental read completed.")
print("--------------------------------------------")
print(f"Bronze rows         : {next_bronze_df.count()}")
print(f"Stored watermark    : {NEXT_WATERMARK}")
print(f"Incremental records : {next_incremental_count}")

assert next_incremental_count == 0, \
    "Expected zero records for unchanged Bronze data."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 27, Finished, Available, Finished, False)

Next Customer incremental read completed.
--------------------------------------------
Bronze rows         : 501
Stored watermark    : 2025-12-19 00:00:00
Incremental records : 0



## Step 12 — Simulate a New Customer Arrival

The incremental framework has successfully demonstrated that previously
processed Customer records are not reprocessed.

A new Customer record is now added to the Bronze layer to simulate the
arrival of new source data.

The new record uses a `created_date` greater than the currently stored
Customer watermark.

Current watermark:

`2025-12-19 00:00:00`

New Customer:

`created_date = 2025-12-20`

The next incremental execution should therefore detect exactly one record.

This test demonstrates the core incremental-load behavior:

`New source data → Watermark filter → Only new records processed`

In [26]:

# ---------------------------------------------------------
# STEP 12A - CREATE ONE NEW BRONZE CUSTOMER
# ---------------------------------------------------------

TEST_CUSTOMER_ID = "CUST_TEST_0001"

new_customer_data = [{
    "customer_id": TEST_CUSTOMER_ID,
    "first_name": "Incremental",
    "last_name": "Test",
    "date_of_birth": "1985-06-15",
    "email": "incremental.test@example.com",
    "phone": "+1-555-010-9001",
    "address": "100 Test Drive",
    "city": "Boston",
    "state": "MA",
    "zip_code": "02108",
    "created_date": "2025-12-20",
    "customer_status": "Active"
}]

bronze_schema = spark.table(SOURCE_TABLE).schema

new_customer_df = spark.createDataFrame(
    new_customer_data,
    schema=bronze_schema
)

display(new_customer_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0873aa0d-7d5d-46a5-b888-81d31faaa0a2)


### Step 12B — Append the New Customer to Bronze

The test Customer record has been validated and is now appended to the
Bronze Customer Delta table.

This simulates a new Customer record arriving from the source system after
the previous incremental load completed.

After the append, Bronze should contain one additional record:

- Previous Bronze count: `501`
- Expected Bronze count: `502`
- New Customer: `CUST_TEST_0001`

In [27]:

# ---------------------------------------------------------
# STEP 12B - APPEND NEW CUSTOMER TO BRONZE
# ---------------------------------------------------------

(
    new_customer_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(SOURCE_TABLE)
)

print("New test Customer appended to Bronze.")
print(f"Customer ID : {TEST_CUSTOMER_ID}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 29, Finished, Available, Finished, False)

New test Customer appended to Bronze.
Customer ID : CUST_TEST_0001


In [28]:
# ---------------------------------------------------------
# STEP 12C - VERIFY NEW BRONZE CUSTOMER
# ---------------------------------------------------------

test_customer_bronze_df = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("customer_id") == TEST_CUSTOMER_ID)
)

test_customer_count = test_customer_bronze_df.count()
bronze_count_after_insert = spark.table(SOURCE_TABLE).count()

display(test_customer_bronze_df)

print("--------------------------------------------")
print(f"Test Customer records : {test_customer_count}")
print(f"Total Bronze rows      : {bronze_count_after_insert}")

assert test_customer_count == 1, \
    "Expected exactly one test Customer in Bronze."
    

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb09e180-3223-4793-81a7-da49e6565b49)

--------------------------------------------
Test Customer records : 1
Total Bronze rows      : 502



## Step 13 — Detect the Newly Arrived Customer

A new Customer record has been appended to the Bronze layer after the
previous Customer watermark.

The Customer control table currently stores:

`last_watermark = 2025-12-19 00:00:00`

The newly arrived Customer contains:

`created_date = 2025-12-20`

The incremental filter now reads the persisted watermark from `etl_control`
and selects only Bronze records whose watermark value is greater than the
last successfully processed watermark.

Expected result:

**1 incremental Customer record**

This demonstrates that the pipeline processes only newly arrived data rather
than rescanning the entire Bronze dataset.

In [29]:
# ---------------------------------------------------------
# STEP 13A - READ CURRENT CUSTOMER WATERMARK
# ---------------------------------------------------------

current_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

CURRENT_WATERMARK = current_config["last_watermark"]
CURRENT_WATERMARK_COLUMN = current_config["watermark_column"]

print("Current Customer watermark loaded.")
print("--------------------------------------------")
print(f"Source name      : {SOURCE_NAME}")
print(f"Watermark column : {CURRENT_WATERMARK_COLUMN}")
print(f"Stored watermark : {CURRENT_WATERMARK}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 31, Finished, Available, Finished, False)

Current Customer watermark loaded.
--------------------------------------------
Source name      : CUSTOMERS
Watermark column : created_date
Stored watermark : 2025-12-19 00:00:00


In [30]:

# ---------------------------------------------------------
# STEP 13B - DETECT NEW CUSTOMER RECORDS
# ---------------------------------------------------------

bronze_customer_df = spark.table(SOURCE_TABLE)

new_incremental_customer_df = (
    bronze_customer_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(CURRENT_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") > F.lit(CURRENT_WATERMARK)
    )
)

new_incremental_count = new_incremental_customer_df.count()

print("Customer incremental detection completed.")
print("--------------------------------------------")
print(f"Total Bronze rows    : {bronze_customer_df.count()}")
print(f"Stored watermark     : {CURRENT_WATERMARK}")
print(f"Incremental records  : {new_incremental_count}")

assert new_incremental_count == 1, \
    f"Expected exactly 1 incremental Customer, found {new_incremental_count}."

display(
    new_incremental_customer_df
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "created_date",
        "customer_status"
    )
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 32, Finished, Available, Finished, False)

Customer incremental detection completed.
--------------------------------------------
Total Bronze rows    : 502
Stored watermark     : 2025-12-19 00:00:00
Incremental records  : 1


SynapseWidget(Synapse.DataFrame, 99ede6ab-df0d-4cdd-80df-23303a36fe36)


## Step 14 — Validate the Incremental Customer

The watermark filter identified exactly one Customer record that arrived after
the previous successful processing position.

Before the record can be promoted to Silver, the incremental dataset must pass
the same data-quality rules used by the Customer processing framework.

For this Customer load, validation checks include:

- `customer_id` must be present.
- `created_date` must contain a valid date.
- Duplicate `customer_id` values within the incremental dataset are rejected.

Only records that pass these checks are eligible for Silver processing.

Expected result for this test:

**Incremental records: 1**  
**Valid records: 1**  
**Rejected records: 0**

In [31]:

# ---------------------------------------------------------
# STEP 14A - VALIDATE NEW INCREMENTAL CUSTOMER
# ---------------------------------------------------------

validated_incremental_df = (
    new_incremental_customer_df

    # Parse created_date for validation
    .withColumn(
        "_parsed_created_date",
        F.to_date(F.col("created_date"))
    )
)

invalid_incremental_df = (
    validated_incremental_df
    .filter(
        F.col("customer_id").isNull()
        | F.col("_parsed_created_date").isNull()
    )
)

valid_incremental_df = (
    validated_incremental_df
    .filter(
        F.col("customer_id").isNotNull()
        & F.col("_parsed_created_date").isNotNull()
    )
)

invalid_count = invalid_incremental_df.count()
valid_pre_dedup_count = valid_incremental_df.count()

print("Incremental Customer validation completed.")
print("--------------------------------------------")
print(f"Incremental records : {new_incremental_count}")
print(f"Valid records       : {valid_pre_dedup_count}")
print(f"Invalid records     : {invalid_count}")

assert (
    valid_pre_dedup_count + invalid_count
    == new_incremental_count
), "Incremental validation reconciliation failed."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 33, Finished, Available, Finished, False)

Incremental Customer validation completed.
--------------------------------------------
Incremental records : 1
Valid records       : 1
Invalid records     : 0


In [32]:

# ---------------------------------------------------------
# STEP 14B - DEDUPLICATE NEW INCREMENTAL CUSTOMER
# ---------------------------------------------------------

duplicate_customer_ids_df = (
    valid_incremental_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .select("customer_id")
)

duplicate_count = duplicate_customer_ids_df.count()

incremental_valid_unique_df = (
    valid_incremental_df
    .join(
        duplicate_customer_ids_df,
        on="customer_id",
        how="left_anti"
    )
)

silver_candidate_count = incremental_valid_unique_df.count()

print("Incremental Customer deduplication completed.")
print("--------------------------------------------")
print(f"Valid input records : {valid_pre_dedup_count}")
print(f"Duplicate keys      : {duplicate_count}")
print(f"Silver candidates   : {silver_candidate_count}")

assert silver_candidate_count == 1, \
    f"Expected 1 Silver candidate, found {silver_candidate_count}."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 34, Finished, Available, Finished, False)

Incremental Customer deduplication completed.
--------------------------------------------
Valid input records : 1
Duplicate keys      : 0
Silver candidates   : 1


In [33]:

display(
    incremental_valid_unique_df.select(
        "customer_id",
        "first_name",
        "last_name",
        "created_date",
        "customer_status"
    )
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2699c3b8-1fd2-4ce0-84f0-860db84a7fae)


## Step 15 — Transform and MERGE the Incremental Customer into Silver

The incremental Customer record has passed validation and deduplication and is
now eligible for Silver processing.

Before writing to Silver, the record is transformed into the standardized
Silver Customer schema.

The Silver load uses a Delta `MERGE` based on the Customer business key:

`customer_id`

The MERGE provides idempotent upsert behavior:

- A Customer not already present in Silver is **INSERTED**.
- An existing Customer with the same business key can be **UPDATED**.
- Existing Silver Customers that are not part of the incremental batch remain unchanged.

For this test:

`CUST_TEST_0001` does not currently exist in Silver.

Expected result:

**Silver rows: 500 → 501**

Only the newly arrived Customer should be inserted.

In [34]:

# ---------------------------------------------------------
# STEP 15A - TRANSFORM INCREMENTAL CUSTOMER FOR SILVER
# ---------------------------------------------------------

incremental_silver_ready_df = (
    incremental_valid_unique_df
    .select(
        F.col("customer_id"),
        F.col("first_name"),
        F.col("last_name"),
        F.to_date("date_of_birth").alias("date_of_birth"),
        F.col("email"),
        F.col("phone"),
        F.col("address"),
        F.col("city"),
        F.col("state"),
        F.col("zip_code"),
        F.to_date("created_date").alias("created_date"),
        F.col("customer_status"),
        F.current_timestamp().alias("_silver_processed_ts")
    )
)

incremental_silver_ready_count = incremental_silver_ready_df.count()

print("Incremental Customer Silver transformation completed.")
print("--------------------------------------------")
print(f"Silver-ready records : {incremental_silver_ready_count}")

assert incremental_silver_ready_count == 1, \
    "Expected exactly one Silver-ready Customer."

incremental_silver_ready_df.printSchema()

display(incremental_silver_ready_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 36, Finished, Available, Finished, False)

Incremental Customer Silver transformation completed.
--------------------------------------------
Silver-ready records : 1
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, e9e0ddca-6281-42d9-9adf-5922b63b5940)

In [35]:
# ---------------------------------------------------------
# STEP 15B - INSPECT SILVER BEFORE MERGE
# ---------------------------------------------------------

silver_before_df = spark.table(TARGET_TABLE)

silver_count_before = silver_before_df.count()

test_customer_silver_before = (
    silver_before_df
    .filter(F.col("customer_id") == TEST_CUSTOMER_ID)
    .count()
)

print("Silver pre-MERGE state.")
print("--------------------------------------------")
print(f"Silver rows              : {silver_count_before}")
print(f"Test Customer occurrences: {test_customer_silver_before}")

assert test_customer_silver_before == 0, \
    "Test Customer already exists in Silver."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 37, Finished, Available, Finished, False)

Silver pre-MERGE state.
--------------------------------------------
Silver rows              : 500
Test Customer occurrences: 0


In [36]:

# ---------------------------------------------------------
# STEP 15C - MERGE INCREMENTAL CUSTOMER INTO SILVER
# ---------------------------------------------------------

silver_delta = DeltaTable.forName(
    spark,
    TARGET_TABLE
)

(
    silver_delta.alias("target")
    .merge(
        incremental_silver_ready_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Incremental Customer Delta MERGE completed.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 38, Finished, Available, Finished, False)

Incremental Customer Delta MERGE completed.


In [37]:
# ---------------------------------------------------------
# STEP 15D - VERIFY SILVER AFTER MERGE
# ---------------------------------------------------------

silver_after_df = spark.table(TARGET_TABLE)

silver_count_after = silver_after_df.count()

test_customer_silver_df = (
    silver_after_df
    .filter(F.col("customer_id") == TEST_CUSTOMER_ID)
)

test_customer_silver_count = test_customer_silver_df.count()

print("Silver post-MERGE verification.")
print("--------------------------------------------")
print(f"Silver rows before       : {silver_count_before}")
print(f"Silver rows after        : {silver_count_after}")
print(f"Net Silver row increase  : {silver_count_after - silver_count_before}")
print(f"Test Customer occurrences: {test_customer_silver_count}")

assert test_customer_silver_count == 1, \
    "Expected exactly one test Customer in Silver."

assert silver_count_after == silver_count_before + 1, \
    "Expected Silver row count to increase by exactly one."

display(test_customer_silver_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 39, Finished, Available, Finished, False)

Silver post-MERGE verification.
--------------------------------------------
Silver rows before       : 500
Silver rows after        : 501
Net Silver row increase  : 1
Test Customer occurrences: 1


SynapseWidget(Synapse.DataFrame, c0fdaf3e-08a0-466e-b316-659943d73a3b)


## Step 16 — Record the Incremental Customer Execution

The incremental Customer record has successfully completed Silver processing.

An operational audit record is now written to `etl_batch_audit` to capture
what occurred during this execution.

For this incremental run:

| Metric | Value |
|---|---:|
| Source records processed | 1 |
| Inserts | 1 |
| Updates | 0 |
| Rejects | 0 |
| Status | SUCCESS |

The audit record provides traceability between the pipeline execution,
the source entity, and the resulting Silver operation.

The Customer watermark is advanced only after successful Silver processing
and audit persistence.

In [38]:

# ---------------------------------------------------------
# STEP 16A - BUILD SECOND-RUN CUSTOMER AUDIT RECORD
# ---------------------------------------------------------

second_run_source_count = new_incremental_count
second_run_insert_count = silver_count_after - silver_count_before
second_run_update_count = 0
second_run_reject_count = invalid_count

second_run_end_time = datetime.now()

second_run_audit_record = [
    (
        BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        second_run_end_time,
        second_run_end_time,
        second_run_source_count,
        second_run_insert_count,
        second_run_update_count,
        second_run_reject_count,
        "SUCCESS",
        None
    )
]

second_run_audit_df = spark.createDataFrame(
    second_run_audit_record,
    schema=audit_schema
)

display(second_run_audit_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3bd504d2-6efb-4c05-b6b9-29469e1b12f4)

In [39]:

# ---------------------------------------------------------
# STEP 16B - PERSIST SECOND-RUN AUDIT RECORD
# ---------------------------------------------------------

(
    second_run_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print("Second-run Customer audit record written successfully.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 41, Finished, Available, Finished, False)

Second-run Customer audit record written successfully.


In [40]:
display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06fccc90-6567-4276-802d-1fb4ea867e7e)

## Step 17 — Advance the Customer Watermark

The incremental Customer record has successfully completed:

1. Incremental detection
2. Data-quality validation
3. Deduplication
4. Silver transformation
5. Delta MERGE
6. Audit persistence

The Customer watermark can now safely advance.

The new watermark is calculated from the maximum successfully processed
`created_date` in the incremental Silver-ready dataset.

Previous watermark:

`2025-12-19 00:00:00`

Expected new watermark:

`2025-12-20 00:00:00`

The watermark is updated only after successful Silver processing and audit
persistence. If processing had failed before this point, the previous watermark
would remain unchanged and the record could be processed again.

In [41]:

# ---------------------------------------------------------
# STEP 17A - CALCULATE SECOND-RUN CUSTOMER WATERMARK
# ---------------------------------------------------------

second_run_watermark = (
    incremental_silver_ready_df
    .agg(
        F.max(
            F.to_timestamp(F.col("created_date"))
        ).alias("new_watermark")
    )
    .first()["new_watermark"]
)

print("Second-run Customer watermark calculated.")
print("--------------------------------------------")
print(f"Previous watermark : {CURRENT_WATERMARK}")
print(f"New watermark      : {second_run_watermark}")

assert second_run_watermark > CURRENT_WATERMARK, \
    "New watermark must be greater than the previous watermark."

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 43, Finished, Available, Finished, False)

Second-run Customer watermark calculated.
--------------------------------------------
Previous watermark : 2025-12-19 00:00:00
New watermark      : 2025-12-20 00:00:00


In [42]:
# ---------------------------------------------------------
# STEP 17B - PERSIST SECOND-RUN CUSTOMER WATERMARK
# ---------------------------------------------------------

control_delta = DeltaTable.forName(
    spark,
    CONTROL_TABLE
)

(
    control_delta.alias("target")
    .update(
        condition=(
            (F.col("target.source_name") == SOURCE_NAME)
            & (F.col("target.is_active") == True)
        ),
        set={
            "last_watermark": F.lit(second_run_watermark),
            "_updated_ts": F.current_timestamp()
        }
    )
)

print("Customer watermark advanced successfully.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 44, Finished, Available, Finished, False)

Customer watermark advanced successfully.


In [43]:
# ---------------------------------------------------------
# STEP 17C - VERIFY UPDATED CUSTOMER WATERMARK
# ---------------------------------------------------------

updated_customer_control_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .select(
        "source_name",
        "watermark_column",
        "last_watermark",
        "load_type",
        "is_active",
        "_updated_ts"
    )
)

display(updated_customer_control_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9da1bc3c-23a1-410d-b206-c191768a1c01)


## Step 18 — Validate the Completed Incremental Cycle

The Customer watermark has successfully advanced to:

`2025-12-20 00:00:00`

The Bronze Customer table currently contains the newly processed Customer
`CUST_TEST_0001`, whose `created_date` is also `2025-12-20`.

The incremental filter is now executed again without adding any additional
source data.

Because the framework processes only records where:

`created_date > last_watermark`

the expected result is:

**0 incremental records**

This verifies that the completed incremental load is restartable and does
not reprocess the Customer that was already successfully promoted to Silver.

In [44]:

# ---------------------------------------------------------
# STEP 18 - FINAL INCREMENTAL RESTART TEST
# ---------------------------------------------------------

final_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

FINAL_WATERMARK = final_config["last_watermark"]

final_bronze_df = spark.table(SOURCE_TABLE)

final_incremental_df = (
    final_bronze_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") > F.lit(FINAL_WATERMARK)
    )
)

final_incremental_count = final_incremental_df.count()

final_silver_count = spark.table(TARGET_TABLE).count()

print("Final Customer incremental restart test.")
print("--------------------------------------------")
print(f"Bronze rows         : {final_bronze_df.count()}")
print(f"Silver rows         : {final_silver_count}")
print(f"Stored watermark    : {FINAL_WATERMARK}")
print(f"Incremental records : {final_incremental_count}")

assert final_incremental_count == 0, \
    "Expected zero records after successful watermark advancement."

print()
print("Customer incremental lifecycle PASSED.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 46, Finished, Available, Finished, False)

Final Customer incremental restart test.
--------------------------------------------
Bronze rows         : 502
Silver rows         : 501
Stored watermark    : 2025-12-20 00:00:00
Incremental records : 0

Customer incremental lifecycle PASSED.



## Step 19 — Execution Identity and Batch ID

Each execution of the Customer incremental process must have its own unique
execution identifier.

The `batch_id` is used to correlate operational information written to
`etl_batch_audit`.

A new `batch_id` must therefore be generated for every Customer processing
execution rather than reused across multiple incremental runs.

### Why this matters

If the same `batch_id` is reused, multiple executions appear to belong to the
same run and operational troubleshooting becomes ambiguous.

The execution model is:

Customer Incremental Execution  
→ Generate unique `batch_id`  
→ Read watermark  
→ Process incremental records  
→ MERGE into Silver  
→ Write audit record  
→ Advance watermark after success

In a later Fabric Pipeline orchestration step, a pipeline-level run identifier
can also be introduced to correlate Customers, Policies, Claims, and Payments
under one parent pipeline execution.

In [45]:

# Unique execution identifier
BATCH_ID = str(uuid.uuid4())



# ---------------------------------------------------------
# EXECUTION IDENTITY
# ---------------------------------------------------------

RUN_START_TIME = datetime.now()
BATCH_ID = str(uuid.uuid4())

print("Customer incremental processing initialized.")
print("---------------------------------------------")
print(f"Source name    : {SOURCE_NAME}")
print(f"Batch ID       : {BATCH_ID}")
print(f"Run start time : {RUN_START_TIME}")
print(f"Pipeline       : {PIPELINE_NAME}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 47, Finished, Available, Finished, False)

Customer incremental processing initialized.
---------------------------------------------
Source name    : CUSTOMERS
Batch ID       : 9fcd7f49-95bb-4b16-89ce-068726aecdeb
Run start time : 2026-08-21 17:43:43.564465
Pipeline       : PL_Insurance_Medallion_ETL


In [46]:
audit_end_time = datetime.now()

audit_record = [
    (
        BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        RUN_START_TIME,
        audit_end_time,
        incremental_count,
        insert_count,
        changed_count,
        reject_count,
        "SUCCESS",
        None
    )
]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5da98a8b-76f4-4937-a78d-23635a439543)


## Step 20 — Handle a No-Data Incremental Run

An incremental pipeline may execute successfully even when no new source
records are available.

This is different from a failure.

When the incremental filter returns zero records, the framework should:

1. Record the execution in `etl_batch_audit`.
2. Set the execution status to `NO_DATA`.
3. Record zero inserts, updates, and rejects.
4. Leave the existing watermark unchanged.
5. End processing successfully.

### Why the Watermark Is Not Updated

The watermark represents the last successfully processed source position.

If no records were processed, there is no new source position to persist.

Therefore:

`incremental_count = 0`

results in:

`status = NO_DATA`

and:

`new watermark = existing watermark`

This distinction allows operations teams to differentiate between:

- `SUCCESS` — records were successfully processed.
- `NO_DATA` — execution succeeded but no new records were available.
- `FAILED` — processing encountered an error.

In [47]:
# ---------------------------------------------------------
# STEP 20A - START CLEAN CUSTOMER EXECUTION
# ---------------------------------------------------------

RUN_START_TIME = datetime.now()
BATCH_ID = str(uuid.uuid4())

print("New Customer execution started.")
print("---------------------------------------------")
print(f"Source name    : {SOURCE_NAME}")
print(f"Batch ID       : {BATCH_ID}")
print(f"Run start time : {RUN_START_TIME}")
print(f"Pipeline       : {PIPELINE_NAME}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 49, Finished, Available, Finished, False)

New Customer execution started.
---------------------------------------------
Source name    : CUSTOMERS
Batch ID       : e139972c-fd5b-4e92-97fb-74da076f57d5
Run start time : 2026-08-21 17:45:55.132423
Pipeline       : PL_Insurance_Medallion_ETL


In [48]:

# ---------------------------------------------------------
# STEP 20B - READ CURRENT CUSTOMER WATERMARK
# ---------------------------------------------------------

run_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

if run_config is None:
    raise ValueError(
        f"No active ETL control configuration found for {SOURCE_NAME}."
    )

RUN_WATERMARK_COLUMN = run_config["watermark_column"]
RUN_WATERMARK = run_config["last_watermark"]

print("Customer execution configuration loaded.")
print("---------------------------------------------")
print(f"Watermark column : {RUN_WATERMARK_COLUMN}")
print(f"Stored watermark : {RUN_WATERMARK}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 50, Finished, Available, Finished, False)

Customer execution configuration loaded.
---------------------------------------------
Watermark column : created_date
Stored watermark : 2025-12-20 00:00:00


In [49]:
# ---------------------------------------------------------
# STEP 20C - READ CUSTOMER INCREMENTAL DATA
# ---------------------------------------------------------

run_bronze_df = spark.table(SOURCE_TABLE)

run_incremental_df = (
    run_bronze_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(RUN_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") > F.lit(RUN_WATERMARK)
    )
)

run_incremental_count = run_incremental_df.count()

print("Customer incremental read completed.")
print("---------------------------------------------")
print(f"Bronze rows         : {run_bronze_df.count()}")
print(f"Stored watermark    : {RUN_WATERMARK}")
print(f"Incremental records : {run_incremental_count}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 51, Finished, Available, Finished, False)

Customer incremental read completed.
---------------------------------------------
Bronze rows         : 502
Stored watermark    : 2025-12-20 00:00:00
Incremental records : 0


In [50]:
# ---------------------------------------------------------
# STEP 20D - HANDLE NO_DATA EXECUTION
# ---------------------------------------------------------

if run_incremental_count == 0:

    RUN_END_TIME = datetime.now()

    no_data_audit_record = [
        (
            BATCH_ID,
            PIPELINE_NAME,
            SOURCE_NAME,
            RUN_START_TIME,
            RUN_END_TIME,
            0,          # source_count
            0,          # insert_count
            0,          # update_count
            0,          # reject_count
            "NO_DATA",
            None
        )
    ]

    no_data_audit_df = spark.createDataFrame(
        no_data_audit_record,
        schema=audit_schema
    )

    display(no_data_audit_df)

else:
    print(
        f"{run_incremental_count} new Customer records found. "
        "Continue normal processing."
    )

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f742c1d-5ad2-478a-932c-23f3c64a31c0)

In [51]:
# ---------------------------------------------------------
# STEP 20E - PERSIST NO_DATA AUDIT RECORD
# ---------------------------------------------------------

if run_incremental_count == 0:

    (
        no_data_audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE)
    )

    print("NO_DATA Customer audit record persisted.")
    print(f"Batch ID : {BATCH_ID}")
    print(f"Status   : NO_DATA")

else:
    print("Audit persistence skipped - records exist for processing.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 53, Finished, Available, Finished, False)

NO_DATA Customer audit record persisted.
Batch ID : e139972c-fd5b-4e92-97fb-74da076f57d5
Status   : NO_DATA


In [52]:
# ---------------------------------------------------------
# STEP 20F - VERIFY NO_DATA EXECUTION
# ---------------------------------------------------------

print("===== CUSTOMER NO_DATA EXECUTION VERIFICATION =====")

print("\nAUDIT RECORD")

final_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

display(final_audit_check_df)


print("\nCONTROL / WATERMARK")

final_control_check_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .select(
        "source_name",
        "watermark_column",
        "last_watermark",
        "load_type",
        "is_active",
        "_updated_ts"
    )
)

display(final_control_check_df)


# Validation
audit_check = final_audit_check_df.first()
control_check = final_control_check_df.first()

assert audit_check["status"] == "NO_DATA", \
    "Expected audit status NO_DATA."

assert audit_check["source_count"] == 0, \
    "Expected source_count = 0."

assert audit_check["insert_count"] == 0, \
    "Expected insert_count = 0."

assert audit_check["update_count"] == 0, \
    "Expected update_count = 0."

assert audit_check["reject_count"] == 0, \
    "Expected reject_count = 0."

assert control_check["last_watermark"] == RUN_WATERMARK, \
    "Watermark changed during NO_DATA execution."

print()
print("Customer NO_DATA execution PASSED.")
print(f"Watermark remains : {control_check['last_watermark']}")


StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 54, Finished, Available, Finished, False)

===== CUSTOMER NO_DATA EXECUTION VERIFICATION =====

AUDIT RECORD


SynapseWidget(Synapse.DataFrame, 649055fd-abeb-4bf4-8504-3a7ff45799af)


CONTROL / WATERMARK


SynapseWidget(Synapse.DataFrame, 0a4434fd-d7ce-48fe-a919-79fb3ed3246b)


Customer NO_DATA execution PASSED.
Watermark remains : 2025-12-20 00:00:00


try:
    # incremental processing
    # validation
    # transformation
    # Delta MERGE
    # audit SUCCESS
    # update watermark

except Exception as e:
    # audit FAILED
    # capture error message
    # DO NOT update watermark
    raise

## Step 21 — Failure Handling and Watermark Protection

Production incremental processing must distinguish between a successful
execution and a failed execution.

If an exception occurs during validation, transformation, or Silver
processing, the framework must:

1. Capture the exception.
2. Write a `FAILED` record to `etl_batch_audit`.
3. Store the error message for operational troubleshooting.
4. Leave the existing watermark unchanged.
5. Re-raise the exception so Fabric marks the notebook execution as failed.

### Critical Design Principle

The watermark must advance only after the Silver write completes
successfully.

Therefore:

**SUCCESS → Audit SUCCESS → Advance watermark**

**NO_DATA → Audit NO_DATA → Keep watermark**

**FAILED → Audit FAILED → Keep watermark**

This prevents failed records from being skipped during the next incremental
execution.

In [54]:
# ---------------------------------------------------------
# STEP 21A - INITIALIZE CONTROLLED FAILURE TEST
# ---------------------------------------------------------

FAILURE_BATCH_ID = str(uuid.uuid4())
FAILURE_START_TIME = datetime.now()

# Capture watermark BEFORE the simulated failure
failure_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

FAILURE_START_WATERMARK = failure_config["last_watermark"]

print("Controlled failure test initialized.")
print("---------------------------------------------")
print(f"Source name       : {SOURCE_NAME}")
print(f"Batch ID          : {FAILURE_BATCH_ID}")
print(f"Starting watermark: {FAILURE_START_WATERMARK}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 56, Finished, Available, Finished, False)

Controlled failure test initialized.
---------------------------------------------
Source name       : CUSTOMERS
Batch ID          : 0cf36726-5703-43f6-94c0-103d3890cca5
Starting watermark: 2025-12-20 00:00:00



### Step 21B — Simulate a Processing Failure

This test intentionally raises a controlled exception to validate the
failure-handling framework.

The test does not modify Bronze or Silver data.

The exception is captured so that a `FAILED` audit record can be created.
The Customer watermark must remain unchanged.

In [55]:

# ---------------------------------------------------------
# STEP 21B - SIMULATE AND CAPTURE PROCESSING FAILURE
# ---------------------------------------------------------

failure_error_message = None
failure_status = None

try:

    print("Starting simulated Customer processing...")

    # Controlled test failure.
    # No Bronze/Silver data is modified.
    raise RuntimeError(
        "SIMULATED_FAILURE: Customer processing test exception."
    )

except Exception as e:

    failure_status = "FAILED"
    failure_error_message = str(e)

    print()
    print("Customer processing failure captured.")
    print("---------------------------------------------")
    print(f"Status        : {failure_status}")
    print(f"Error message : {failure_error_message}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 57, Finished, Available, Finished, False)

Starting simulated Customer processing...

Customer processing failure captured.
---------------------------------------------
Status        : FAILED
Error message : SIMULATED_FAILURE: Customer processing test exception.



### Step 21C — Create the FAILED Audit Record

The simulated exception has been captured.

This step creates an audit record with:

- `status = FAILED`
- zero inserts
- zero updates
- zero rejects
- the captured error message

The watermark is not changed during failure handling.

In [56]:

# ---------------------------------------------------------
# STEP 21C - BUILD FAILED CUSTOMER AUDIT RECORD
# ---------------------------------------------------------

FAILURE_END_TIME = datetime.now()

failed_audit_record = [
    (
        FAILURE_BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        FAILURE_START_TIME,
        FAILURE_END_TIME,
        0,                        # source_count
        0,                        # insert_count
        0,                        # update_count
        0,                        # reject_count
        failure_status,
        failure_error_message
    )
]

failed_audit_df = spark.createDataFrame(
    failed_audit_record,
    schema=audit_schema
)

display(failed_audit_df)

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c539bd9-b57d-4c4a-9e5f-712cb24fe972)

In [57]:
# ---------------------------------------------------------
# STEP 21D - PERSIST FAILED AUDIT RECORD
# ---------------------------------------------------------

(
    failed_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print("FAILED Customer audit record persisted.")
print(f"Batch ID : {FAILURE_BATCH_ID}")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 59, Finished, Available, Finished, False)

FAILED Customer audit record persisted.
Batch ID : 0cf36726-5703-43f6-94c0-103d3890cca5


In [58]:
# ---------------------------------------------------------
# STEP 21E - VERIFY FAILED EXECUTION
# ---------------------------------------------------------

failure_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == FAILURE_BATCH_ID)
)

failure_control_check = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

display(failure_audit_check_df)

print("---------------------------------------------")
print(f"Starting watermark : {FAILURE_START_WATERMARK}")
print(f"Current watermark  : {failure_control_check['last_watermark']}")

assert failure_audit_check_df.first()["status"] == "FAILED", \
    "Expected FAILED audit status."

assert failure_control_check["last_watermark"] == FAILURE_START_WATERMARK, \
    "Watermark changed during failed execution."

print()
print("Customer FAILED execution handling PASSED.")

StatementMeta(, bfa2d1a0-35cc-4ac0-80fe-4e4b85d83cde, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2cbb3e2f-f852-4a98-af75-ed2fe3a68dc6)

---------------------------------------------
Starting watermark : 2025-12-20 00:00:00
Current watermark  : 2025-12-20 00:00:00

Customer FAILED execution handling PASSED.




---

# NB_02 — Customer Incremental Processing: Implementation Summary

## What This Notebook Implements

This notebook implements and validates the complete incremental
**Bronze-to-Silver processing pattern for Customers**.

### Architecture

`LH_Bronze.dbo.bronze_customers`  
↓  
Read Customer configuration from `etl_control`  
↓  
Read last successful watermark  
↓  
Incremental filter using `created_date`  
↓  
Data-quality validation  
↓  
Business-key deduplication using `customer_id`  
↓  
Silver transformation  
↓  
Delta MERGE  
↓  
`LH_Silver.dbo.silver_customers`  
↓  
Write execution result to `etl_batch_audit`  
↓  
Advance watermark only after successful processing

---

## Processing Components

| Component | Implementation |
|---|---|
| Source | `LH_Bronze.dbo.bronze_customers` |
| Target | `LH_Silver.dbo.silver_customers` |
| Business Key | `customer_id` |
| Watermark Column | `created_date` |
| Control Table | `LH_Silver.dbo.etl_control` |
| Audit Table | `LH_Silver.dbo.etl_batch_audit` |
| Storage Format | Delta |
| Write Pattern | Delta MERGE |
| Load Pattern | Incremental |

---

## Execution States

The framework supports three operational outcomes.

### SUCCESS

New records are detected and successfully processed.

`Incremental Read → Validation → Deduplication → Transformation → Delta MERGE → Audit SUCCESS → Advance Watermark`

### NO_DATA

No records exist beyond the current watermark.

`Incremental Read → 0 Records → Audit NO_DATA → Keep Existing Watermark`

### FAILED

An exception occurs during processing.

`Processing → Exception → Audit FAILED → Capture Error → Keep Existing Watermark`

---

## Watermark Safety Principle

The watermark represents the **last successfully processed source position**.

Therefore, the watermark must never advance before successful Silver
processing.

This provides restartability:

**Successful processing → advance watermark**

**No data → keep watermark**

**Failed processing → keep watermark**

A failed execution can therefore safely retry the same source records during
the next execution.

---

## Incremental Test Performed

The Customer framework was tested by inserting:

`CUST_TEST_0001`

into Bronze with:

`created_date = 2025-12-20`

The previous Customer watermark was:

`2025-12-19`

The incremental process detected exactly one new record and merged it into
Silver.

The Customer watermark was then advanced to:

`2025-12-20`

A subsequent execution detected:

`0 incremental records`

confirming that the previously processed Customer was not processed again.

---

## Final Validated State

**Bronze Customers:** 502  
**Silver Customers:** 501  
**Customer Watermark:** `2025-12-20 00:00:00`

The difference between Bronze and Silver is expected because duplicate
Customer records were identified during data-quality processing.

---

## Patterns Proven by This Notebook

This implementation demonstrates:

- Metadata-driven incremental processing
- Watermark-based change detection
- Data-quality validation
- Business-key deduplication
- Bronze-to-Silver transformation
- Delta MERGE
- Insert/update detection
- Execution auditing
- Unique execution/batch identification
- SUCCESS processing
- NO_DATA processing
- FAILED processing
- Error-message capture
- Watermark protection
- Restart-safe incremental execution

---

## Reusable Architecture

`NB_02` serves as the reference implementation for the remaining insurance
entities.

The same framework will be extended to:

- `NB_03 — Policies Incremental Bronze-to-Silver`
- `NB_04 — Claims Incremental Bronze-to-Silver`
- `NB_05 — Payments Incremental Bronze-to-Silver`

Each entity will reuse the same control, audit, watermark, Delta MERGE, and
execution-state architecture while implementing entity-specific business
keys, watermark columns, validation rules, and transformations.
